In [0]:
# dbutils.fs.ls("/Volumes/adb_caresync/landing/landing_volume/neon_postgres/hospitals/2026-11-05_03:40:30/")

In [0]:
# Create widgets
dbutils.widgets.text("source_system", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("source_file_format", "")

# Get widget values into variables
source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
landing_path = f"/Volumes/adb_caresync/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/"

# Check that the landing folder exists and has files.
# dbutils.fs.ls raises an error if the folder (or any parent folder) is missing.
try:
    landing_files = dbutils.fs.ls(landing_path)
except Exception as e:
    if "FileNotFoundException" in str(e) or "does not exist" in str(e).lower():
        landing_files = []
    else:
        raise  # any other error (e.g. permissions) should still fail the notebook

# Stop the notebook cleanly (no failure) when there is nothing to load.
# exit() is kept outside the try/except so it is not caught by it.
if not landing_files:
    print(f"No new files received at {landing_path}. Exiting notebook.")
    dbutils.notebook.exit("0")

if source_file_format == "csv":
    landing_df = spark.read.format("csv").option("header", True).option("inferSchema", True).load(landing_path)
else:
    landing_df = spark.read.format("parquet").load(landing_path)

In [0]:
# display(landing_df)


In [0]:
from pyspark.sql.functions import lit, current_timestamp

updated_df = landing_df.withColumn("landing_timestamp", lit(landing_timestamp)) \
                       .withColumn("insert_timestamp", current_timestamp())

In [0]:
# display(updated_df)
updated_df.write.mode("append").saveAsTable(f"adb_caresync.bronze.{table_name}")

In [0]:
# %sql
# select * from adb_caresync.bronze.hospitals